# Word-Level Descriptors

Rerunnable version of the descriptor notebook using the current cluster paths. It reads the transcript Excel files from `/scratch/aniluchavez/ConvoDATAS/Transcripts`, includes the new five patients, and writes outputs to `../results/wordleveldescriptors`.


In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Figure font/export settings. Keep SVG text editable in Illustrator.
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Nimbus Sans', 'Arial', 'DejaVu Sans'],
    'svg.fonttype': 'none',   # keep SVG text as text instead of paths
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# Paths
PROJECT_DIR = Path('/scratch/aniluchavez/hippocampal-speaker-semantics')
TRANSCRIPTS_ROOT = Path('/scratch/aniluchavez/ConvoDATAS/Transcripts')
OUT_DIR = PROJECT_DIR / 'results' / 'wordleveldescriptors'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Patients included in the rerun. The final five are the newly added patients.
PATIENTS = [
    'PTYEU_task147',
    'PTYEV_task37',
    'PTYEY_task86',
    'PTYEZ_task60',
    'PTYFA_task25',
    'PTYFC_task28',
    'PTYFF_task17',
    'PTYFG_task18',
    'PTYFI_task81',
    'PTYFK_task40',
    'PTYFM_task104',
    'PTYFP_task88',
    'PTYFR_task91',
    'PTYFS_task95',
    'PTYFU_task224',
]
NEW5 = {'PTYFM_task104', 'PTYFP_task88', 'PTYFR_task91', 'PTYFS_task95', 'PTYFU_task224'}
TARGET_SPEAKER = 'Speaker1'

print(f'Transcripts directory: {TRANSCRIPTS_ROOT}')
print(f'Output directory: {OUT_DIR}')
print(f'Patients: {len(PATIENTS)} total, {len(NEW5)} new')


In [ ]:
def find_excel(root: Path, patient_id: str) -> Path:
    matches = sorted(
        p for p in root.glob(f'{patient_id}*.xlsx')
        if not p.name.startswith(('~$', '._')) and p.exists()
    )
    if not matches:
        raise FileNotFoundError(f'No transcript .xlsx found for {patient_id}: {root}')

    # Prefer the manually updated Newest transcript when both New and Newest exist.
    matches = sorted(matches, key=lambda p: ('Newest' not in p.name, p.name))
    return matches[0]


def speaker_columns(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if re.match(r'^Speaker\d+$', str(c))]


def normalize_token(token: object) -> str:
    text = str(token).strip().lower()
    text = re.sub(r"^[^\w']+|[^\w']+$", '', text)
    return text


def load_long_words(excel_path: Path, patient_id: str) -> pd.DataFrame:
    df = pd.read_excel(excel_path)
    spk_cols = speaker_columns(df)
    if not spk_cols:
        raise ValueError(f'No Speaker# columns found in {excel_path}; columns={list(df.columns)}')

    id_vars = [c for c in ['onset', 'offset', 'Duration', 'regress_dur'] if c in df.columns]
    long_df = df.melt(
        id_vars=id_vars,
        value_vars=spk_cols,
        var_name='speaker',
        value_name='word_token',
    )
    long_df = long_df.dropna(subset=['word_token']).copy()
    long_df['word'] = long_df['word_token']
    long_df['patient_id'] = patient_id
    long_df['is_new5'] = patient_id in NEW5
    long_df['source_excel'] = str(excel_path)

    if 'Duration' in long_df.columns:
        long_df['Duration'] = pd.to_numeric(long_df['Duration'], errors='coerce')

    long_df['token_norm'] = long_df['word_token'].map(normalize_token)
    long_df = long_df[long_df['token_norm'] != ''].reset_index(drop=True)
    return long_df


In [ ]:
def wpm_summary(words: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for patient_id, sub in words.groupby('patient_id', sort=False):
        dur = pd.to_numeric(sub['Duration'], errors='coerce').dropna()
        total_words = int(len(dur))
        total_dur_ms = float(dur.sum())
        mean_dur_ms = float(dur.mean()) if total_words else np.nan
        rows.append({
            'patient_id': patient_id,
            'is_new5': patient_id in NEW5,
            'n_words': total_words,
            'total_dur_ms': total_dur_ms,
            'total_dur_min': total_dur_ms / 60000.0,
            'mean_dur_ms': mean_dur_ms,
            'median_dur_ms': float(dur.median()) if total_words else np.nan,
            'wpm_by_sum': total_words / (total_dur_ms / 60000.0) if total_dur_ms > 0 else np.nan,
            'wpm_by_mean': 60000.0 / mean_dur_ms if mean_dur_ms > 0 else np.nan,
        })
    return pd.DataFrame(rows)


def vocab_summary(words: pd.DataFrame, target_speaker: str) -> tuple[pd.DataFrame, dict]:
    rows = []
    overall_self_vocab = set()
    overall_other_vocab = set()
    shared_counter = Counter()

    for patient_id, sub in words.groupby('patient_id', sort=False):
        self_tokens = sub.loc[sub['speaker'] == target_speaker, 'token_norm'].tolist()
        other_tokens = sub.loc[sub['speaker'] != target_speaker, 'token_norm'].tolist()
        unique_self = set(self_tokens)
        unique_other = set(other_tokens)
        shared = unique_self & unique_other

        overall_self_vocab.update(unique_self)
        overall_other_vocab.update(unique_other)
        shared_counter.update([t for t in self_tokens + other_tokens if t in shared])

        rows.append({
            'patient_id': patient_id,
            'is_new5': patient_id in NEW5,
            'n_self_tokens': len(self_tokens),
            'n_other_tokens': len(other_tokens),
            'n_unique_self': len(unique_self),
            'n_unique_other': len(unique_other),
            'ttr_self': len(unique_self) / len(self_tokens) if self_tokens else np.nan,
            'ttr_other': len(unique_other) / len(other_tokens) if other_tokens else np.nan,
            'n_shared_unique': len(shared),
            'n_shared_occurrences': sum(1 for t in self_tokens + other_tokens if t in shared),
        })

    overall = {
        'target_speaker': target_speaker,
        'n_patients': int(words['patient_id'].nunique()),
        'n_new5_patients': int(words.loc[words['is_new5'], 'patient_id'].nunique()),
        'overall_unique_self': len(overall_self_vocab),
        'overall_unique_other': len(overall_other_vocab),
        'overall_shared_unique': len(overall_self_vocab & overall_other_vocab),
        'top_shared_tokens': shared_counter.most_common(50),
    }
    return pd.DataFrame(rows), overall


def turn_summary(words: pd.DataFrame, target_speaker: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_turns = []
    for patient_id, sub in words.groupby('patient_id', sort=False):
        sort_cols = [c for c in ['onset', 'offset'] if c in sub.columns]
        long_df = sub.sort_values(sort_cols).reset_index(drop=True) if sort_cols else sub.reset_index(drop=True)
        long_df['speaker_shift'] = long_df['speaker'] != long_df['speaker'].shift()
        long_df['turn_id'] = long_df['speaker_shift'].cumsum()

        grouped = (
            long_df.groupby('turn_id', as_index=False)
            .agg(speaker=('speaker', 'first'), onset=('onset', 'min'), n_words=('word', 'count'))
        )
        grouped['patient_id'] = patient_id
        grouped['is_new5'] = patient_id in NEW5
        grouped['role'] = np.where(grouped['speaker'] == target_speaker, 'self', 'other')
        all_turns.append(grouped)

    turns = pd.concat(all_turns, ignore_index=True)
    role_summary = (
        turns.groupby(['role'], as_index=False)['n_words']
        .agg(n_turns='count', mean_n_words='mean', median_n_words='median', std_n_words='std')
    )
    return turns, role_summary


In [ ]:
long_frames = []
manifest = []

for patient_id in PATIENTS:
    excel = find_excel(TRANSCRIPTS_ROOT, patient_id)
    print(f'reading {patient_id}: {excel}')
    long_df = load_long_words(excel, patient_id)
    long_frames.append(long_df)
    manifest.append({
        'patient_id': patient_id,
        'is_new5': patient_id in NEW5,
        'source_excel': str(excel),
        'n_words': int(len(long_df)),
        'speakers': sorted(long_df['speaker'].unique().tolist()),
    })

words = pd.concat(long_frames, ignore_index=True)
manifest_df = pd.DataFrame(manifest)

words.to_csv(OUT_DIR / 'all_patients_words_durations.csv', index=False)
manifest_df.to_csv(OUT_DIR / 'manifest.csv', index=False)

print(f'Loaded {words.patient_id.nunique()} patients')
print(f'Total word rows: {len(words):,}')
manifest_df[['patient_id', 'is_new5', 'n_words']]


In [ ]:
wpm = wpm_summary(words)
vocab, overall_vocab = vocab_summary(words, TARGET_SPEAKER)
turns, turn_role_summary = turn_summary(words, TARGET_SPEAKER)

wpm.to_csv(OUT_DIR / 'per_patient_wpm_summary.csv', index=False)
vocab.to_csv(OUT_DIR / 'per_patient_role_vocab_summary.csv', index=False)
turns.to_csv(OUT_DIR / 'all_patients_turn_summary.csv', index=False)
turn_role_summary.to_csv(OUT_DIR / 'turn_summary_by_role.csv', index=False)
(OUT_DIR / 'overall_role_vocab_summary.json').write_text(json.dumps(overall_vocab, indent=2) + '\n')

print(f'Turn rows: {len(turns):,}')
print(f'New-five patients in word table: {words.loc[words.is_new5, "patient_id"].nunique()}')
print(f'New-five patients in turn table: {turns.loc[turns.is_new5, "patient_id"].nunique()}')
wpm


In [ ]:
# Per-patient speaking/listening word-count pie charts
# speaking = Speaker1/self words; listening = all other-speaker words.
# Keep vector-export text editable in Illustrator.
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

word_role_counts = (
    words.assign(role=np.where(words['speaker'] == TARGET_SPEAKER, 'speaking', 'listening'))
    .groupby(['patient_id', 'role'])
    .size()
    .unstack(fill_value=0)
    .reindex(PATIENTS)
)

for col in ['speaking', 'listening']:
    if col not in word_role_counts.columns:
        word_role_counts[col] = 0
word_role_counts = word_role_counts[['speaking', 'listening']]
word_role_counts.to_csv(OUT_DIR / 'per_patient_speaking_listening_word_counts.csv')

ncols = 4
nrows = int(np.ceil(len(word_role_counts) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(9.4, 1.95 * nrows))
axes = np.asarray(axes).ravel()
colors = {'speaking': '#F07C6E', 'listening': '#6C63F6'}

for ax, (patient_id, row) in zip(axes, word_role_counts.iterrows()):
    vals = [int(row['speaking']), int(row['listening'])]
    labels = ['speaking', 'listening']
    wedges, texts, autotexts = ax.pie(
        vals,
        colors=[colors['speaking'], colors['listening']],
        startangle=90,
        counterclock=True,
        autopct=lambda pct: f'{int(round(pct * sum(vals) / 100.0))}' if pct > 0 else '',
        pctdistance=0.58,
        radius=1.08,
        textprops={'fontsize': 11, 'color': 'white'},
        wedgeprops={'edgecolor': 'black', 'linewidth': 1.0},
    )
    short_label = patient_id.replace('PTY', 'Y').split('_')[0]
    ax.text(0.96, -0.10, short_label, ha='left', va='center', fontsize=14)
    ax.set_aspect('equal')
    ax.set_axis_off()

for ax in axes[len(word_role_counts):]:
    ax.axis('off')

legend_handles = [
    plt.Line2D([0], [0], marker='o', linestyle='', markersize=12,
               markerfacecolor=colors['speaking'], markeredgecolor='black', label='speaking'),
    plt.Line2D([0], [0], marker='o', linestyle='', markersize=12,
               markerfacecolor=colors['listening'], markeredgecolor='black', label='listening'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False, fontsize=14)
fig.subplots_adjust(left=0.015, right=0.985, top=0.99, bottom=0.10, wspace=-0.10, hspace=0.04)

fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies.png', dpi=300)
fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies.eps', format='eps')
fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies.svg', format='svg')
plt.show()

word_role_counts


In [ ]:
# Per-patient speaking/listening word-count pie charts, 5 rows x 3 columns
# speaking = Speaker1/self words; listening = all other-speaker words.
# Keep vector-export text editable in Illustrator.
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

word_role_counts = (
    words.assign(role=np.where(words['speaker'] == TARGET_SPEAKER, 'speaking', 'listening'))
    .groupby(['patient_id', 'role'])
    .size()
    .unstack(fill_value=0)
    .reindex(PATIENTS)
)

for col in ['speaking', 'listening']:
    if col not in word_role_counts.columns:
        word_role_counts[col] = 0
word_role_counts = word_role_counts[['speaking', 'listening']]
word_role_counts.to_csv(OUT_DIR / 'per_patient_speaking_listening_word_counts.csv')

ncols = 3
nrows = 5
fig, axes = plt.subplots(nrows, ncols, figsize=(6.8, 1.75 * nrows))
axes = np.asarray(axes).ravel()
colors = {'speaking': '#F07C6E', 'listening': '#6C63F6'}

for ax, (patient_id, row) in zip(axes, word_role_counts.iterrows()):
    vals = [int(row['speaking']), int(row['listening'])]
    labels = ['speaking', 'listening']
    wedges, texts, autotexts = ax.pie(
        vals,
        colors=[colors['speaking'], colors['listening']],
        startangle=90,
        counterclock=True,
        autopct=lambda pct: f'{int(round(pct * sum(vals) / 100.0))}' if pct > 0 else '',
        pctdistance=0.58,
        radius=1.08,
        textprops={'fontsize': 11, 'color': 'white'},
        wedgeprops={'edgecolor': 'black', 'linewidth': 1.0},
    )
    short_label = patient_id.replace('PTY', 'Y').split('_')[0]
    ax.text(0.96, -0.10, short_label, ha='left', va='center', fontsize=14)
    ax.set_aspect('equal')
    ax.set_axis_off()

for ax in axes[len(word_role_counts):]:
    ax.axis('off')

legend_handles = [
    plt.Line2D([0], [0], marker='o', linestyle='', markersize=12,
               markerfacecolor=colors['speaking'], markeredgecolor='black', label='speaking'),
    plt.Line2D([0], [0], marker='o', linestyle='', markersize=12,
               markerfacecolor=colors['listening'], markeredgecolor='black', label='listening'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False, fontsize=14)
fig.subplots_adjust(left=0.02, right=0.98, top=0.99, bottom=0.08, wspace=-0.08, hspace=0.00)

fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies_5rows_3cols.png', dpi=300)
fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies_5rows_3cols.eps', format='eps')
fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies_5rows_3cols.svg', format='svg')
plt.show()

word_role_counts


In [ ]:
# Per-patient speaking/listening word-count pie charts, 5 x 3 layout
# speaking = Speaker1/self words; listening = all other-speaker words.
# Keep vector-export text editable in Illustrator.
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

word_role_counts = (
    words.assign(role=np.where(words['speaker'] == TARGET_SPEAKER, 'speaking', 'listening'))
    .groupby(['patient_id', 'role'])
    .size()
    .unstack(fill_value=0)
    .reindex(PATIENTS)
)

for col in ['speaking', 'listening']:
    if col not in word_role_counts.columns:
        word_role_counts[col] = 0
word_role_counts = word_role_counts[['speaking', 'listening']]
word_role_counts.to_csv(OUT_DIR / 'per_patient_speaking_listening_word_counts.csv')

ncols = 5
nrows = 3
fig, axes = plt.subplots(nrows, ncols, figsize=(8.4, 1.75 * nrows))
axes = np.asarray(axes).ravel()
colors = {'speaking': '#F07C6E', 'listening': '#6C63F6'}

for ax, (patient_id, row) in zip(axes, word_role_counts.iterrows()):
    vals = [int(row['speaking']), int(row['listening'])]
    labels = ['speaking', 'listening']
    wedges, texts, autotexts = ax.pie(
        vals,
        colors=[colors['speaking'], colors['listening']],
        startangle=90,
        counterclock=True,
        autopct=lambda pct: f'{int(round(pct * sum(vals) / 100.0))}' if pct > 0 else '',
        pctdistance=0.58,
        radius=1.08,
        textprops={'fontsize': 11, 'color': 'white'},
        wedgeprops={'edgecolor': 'black', 'linewidth': 1.0},
    )
    short_label = patient_id.replace('PTY', 'Y').split('_')[0]
    ax.text(0.96, -0.10, short_label, ha='left', va='center', fontsize=14)
    ax.set_aspect('equal')
    ax.set_axis_off()

for ax in axes[len(word_role_counts):]:
    ax.axis('off')

legend_handles = [
    plt.Line2D([0], [0], marker='o', linestyle='', markersize=12,
               markerfacecolor=colors['speaking'], markeredgecolor='black', label='speaking'),
    plt.Line2D([0], [0], marker='o', linestyle='', markersize=12,
               markerfacecolor=colors['listening'], markeredgecolor='black', label='listening'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False, fontsize=14)
fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.12, wspace=-0.18, hspace=0.02)

fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies_5x3.png', dpi=300)
fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies_5x3.eps', format='eps')
fig.savefig(OUT_DIR / 'per_patient_speaking_listening_word_pies_5x3.svg', format='svg')
plt.show()

word_role_counts


In [ ]:
# Figure helpers. Font/SVG export settings are configured in the setup cell.
# Word duration histogram
fig, ax = plt.subplots(figsize=(10, 8))
durations = pd.to_numeric(words['Duration'], errors='coerce').dropna()
counts, bin_edges, patches = ax.hist(durations, bins=150, color='#cccccc', edgecolor='black')
ax.set_xlim(0, 1000)
ax.set_xlabel('Duration (ms)', fontsize=24)
ax.set_ylabel('Count (x1000)', fontsize=24)

# Show y-axis tick labels in thousands while keeping the histogram counts unchanged.
ymax = counts[bin_edges[:-1] <= 1000].max()
ytick_top = np.ceil(ymax / 2000) * 2000
yticks = np.arange(0, ytick_top + 1, 2000)
ax.set_yticks(yticks)
ax.set_yticklabels([f'{int(t / 1000)}' for t in yticks])

ax.tick_params(axis='both', which='major', labelsize=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / 'word_duration_histogram.png', dpi=300)
fig.savefig(OUT_DIR / 'word_duration_histogram.eps', format='eps')
fig.savefig(OUT_DIR / 'word_duration_histogram.svg', format='svg')
plt.show()


In [ ]:
# Speaker-turn histograms: pooled plus separated by speaking/listening
# Here speaking = Speaker1/self turns; listening = all other-speaker turns.
turn_hist_specs = [
    ('all', turns, 'Words per speaker turn', 'Number of turns', '#CC5500', 'all_patients_turn_histogram'),
    ('speaking', turns[turns['role'] == 'self'], 'Words per speaking turn', 'Number of turns', '#FF0000', 'speaking_turn_histogram'),
    ('listening', turns[turns['role'] == 'other'], 'Words per listening turn', 'Number of turns', '#0000FF', 'listening_turn_histogram'),
]

# n_words is integer-valued, so use integer-aligned bins.
# This avoids empty half-word bins from using too many continuous bins.
turn_bins = np.arange(0.5, 61.5, 1)

for label, plot_df, xlabel, ylabel, color, stem in turn_hist_specs:
    fig, ax = plt.subplots(figsize=(1.5949, 1.3952))
    ax.hist(plot_df['n_words'], bins=turn_bins, edgecolor='black', linewidth=0.35, color=color)
    ax.set_xlim(0, 60)
    ax.set_xlabel(xlabel, fontsize=6)
    ax.set_ylabel(ylabel, fontsize=6)
    ax.tick_params(axis='both', which='major', labelsize=5, length=2, width=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.6)
    ax.spines['bottom'].set_linewidth(0.6)
    ax.set_title(f'{label.capitalize()} turns', fontsize=7, pad=1)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f'{stem}.png', dpi=300)
    fig.savefig(OUT_DIR / f'{stem}.eps', format='eps')
    fig.savefig(OUT_DIR / f'{stem}.svg', format='svg')
    plt.show()


In [ ]:
# Speaker-turn cumulative density
all_sizes = turns['n_words'].to_numpy()
sorted_sizes = np.sort(all_sizes)
cumulative = np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes)
q50, q75 = np.percentile(all_sizes, [50, 75])

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(sorted_sizes, cumulative, color='#CC5500')
ax.set_xlim(0, 125)
ax.set_xlabel('Words per speaker turn', fontsize=24)
ax.set_ylabel('Cumulative fraction of turns', fontsize=24)
for q, label, y in [(q50, '50%', 0.12), (q75, '75%', 0.05)]:
    ax.axvline(x=q, color='gray', linestyle='--', alpha=0.7)
    ax.text(q + 2, y, f'{label}: {q:.0f}', rotation=90, va='bottom', color='gray', fontsize=20)
ax.tick_params(axis='both', which='major', labelsize=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
fig.tight_layout()
fig.savefig(OUT_DIR / 'all_patients_turn_cdf.png', dpi=300)
fig.savefig(OUT_DIR / 'all_patients_turn_cdf.eps', format='eps')
fig.savefig(OUT_DIR / 'all_patients_turn_cdf.svg', format='svg')
plt.show()


In [ ]:
print('Saved outputs:')
for path in sorted(OUT_DIR.iterdir()):
    print(path)
